In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import requests
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [14]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 🔹 Load your raw dataset
df = pd.read_csv("hospital_1.csv")  # Change to your raw CSV name

# 🔹 Define DASS Depression question columns
depression_qs = ['Q3A', 'Q5A', 'Q10A', 'Q13A', 'Q16A', 'Q17A', 'Q21A',
                 'Q24A', 'Q26A', 'Q31A', 'Q34A', 'Q37A', 'Q38A', 'Q42A']
depression_df = df[depression_qs].copy().apply(pd.to_numeric, errors='coerce') - 1
depression_df['Total_Count'] = depression_df.sum(axis=1)

# 🔹 Compute Condition labels
def get_condition(score):
    if score <= 9:
        return 'Normal'
    elif 10 <= score <= 13:
        return 'Mild'
    elif 14 <= score <= 20:
        return 'Moderate'
    elif 21 <= score <= 27:
        return 'Severe'
    else:
        return 'Extremely Severe'

df['Condition'] = depression_df['Total_Count'].apply(get_condition)

# 🔹 Encode features
df_encoded = df.copy()
label_encoders = {}
target_column = 'Condition'

for col in df_encoded.select_dtypes(include='object').columns:
    if col != target_column:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

# 🔹 Encode target
le_target = LabelEncoder()
df_encoded[target_column] = le_target.fit_transform(df_encoded[target_column])

# 🔹 Train/Val split
X = df_encoded.drop(columns=[target_column])
y = df_encoded[target_column]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

print("✅ Dataset prepared! Shapes:")
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")


✅ Dataset prepared! Shapes:
X_train: (10299, 41), X_val: (2575, 41)


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.model(x)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))
model = MLP(input_dim, num_classes)

# Basic training (1-3 epochs is enough)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

X_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train.values, dtype=torch.long)

model.train()
for epoch in range(3):
    optimizer.zero_grad()
    output = model(X_tensor)
    loss = criterion(output, y_tensor)
    loss.backward()
    optimizer.step()

print("✅ Model trained on clean data.")


✅ Model trained on clean data.


In [16]:
from sklearn.metrics import accuracy_score, classification_report

def flatten_gradients(model):
    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1).cpu())
    return torch.cat(grads)

def mitm_modify_gradient(gradient):
    print("\n💀 [MITM] Modifying raw gradients...")
    tampered = gradient.clone()
    tampered += torch.randn_like(tampered) * 5  # Heavy noise
    return tampered

def apply_fake_aggregation(model, modified_grads):
    idx = 0
    for param in model.parameters():
        if param.grad is not None:
            numel = param.numel()
            new_grad = modified_grads[idx:idx+numel].view_as(param.data)
            param.data -= 0.01 * new_grad  # Fake global update
            idx += numel

def evaluate_model(model, X_val_tensor, y_val_tensor):
    model.eval()
    with torch.no_grad():
        preds = model(X_val_tensor).argmax(dim=1)
    acc = accuracy_score(y_val_tensor, preds)
    print(f"\n📉 [AFTER ATTACK] Model Accuracy: {acc:.4f}")
    print(classification_report(y_val_tensor, preds))

def run_mitm_attack_simulation(model, X_train, y_train, X_val, y_val):
    print("[CLIENT] 🧠 Backpropagation to extract gradients...")
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    X_tensor = torch.tensor(X_train.values, dtype=torch.float32)
    y_tensor = torch.tensor(y_train.values, dtype=torch.long)
    X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

    model.train()
    optimizer.zero_grad()
    output = model(X_tensor)
    loss = criterion(output, y_tensor)
    loss.backward()

    raw_grads = flatten_gradients(model)
    tampered_grads = mitm_modify_gradient(raw_grads)
    apply_fake_aggregation(model, tampered_grads)
    evaluate_model(model, X_val_tensor, y_val_tensor)

# ✅ Run it
run_mitm_attack_simulation(model, X_train, y_train, X_val, y_val)


[CLIENT] 🧠 Backpropagation to extract gradients...

💀 [MITM] Modifying raw gradients...

📉 [AFTER ATTACK] Model Accuracy: 0.1876
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       880
           1       0.00      0.00      0.00       243
           2       0.18      0.92      0.31       472
           3       0.23      0.09      0.13       568
           4       0.00      0.00      0.00       412

    accuracy                           0.19      2575
   macro avg       0.08      0.20      0.09      2575
weighted avg       0.09      0.19      0.08      2575



C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo